In [1]:
# Importing the common file shared between both problems
from common import *

## Hyperparameters

In [2]:
epochs = 10
batch_size = 64
image_size = 32
num_classes = 100
learning_rate = 0.001
input_size = (1, 3, 32, 32) # (batch_dim, RGB channels, image size)

# Initializing all 16 different configurations
configs = [
    {'name': 'patch4_embed256_layers4_heads4', 'patch': 4, 'embed': 256, 'blocks': 4, 'heads': 4},
    {'name': 'patch4_embed256_layers4_heads8', 'patch': 4, 'embed': 256, 'blocks': 4, 'heads': 8},
    {'name': 'patch4_embed256_layers8_heads4', 'patch': 4, 'embed': 256, 'blocks': 8, 'heads': 4},
    {'name': 'patch4_embed256_layers8_heads8', 'patch': 4, 'embed': 256, 'blocks': 8, 'heads': 8},
    {'name': 'patch4_embed512_layers4_heads4', 'patch': 4, 'embed': 512, 'blocks': 4, 'heads': 4},
    {'name': 'patch4_embed512_layers4_heads8', 'patch': 4, 'embed': 512, 'blocks': 4, 'heads': 8},
    {'name': 'patch4_embed512_layers8_heads4', 'patch': 4, 'embed': 512, 'blocks': 8, 'heads': 4},
    {'name': 'patch4_embed512_layers8_heads8', 'patch': 4, 'embed': 512, 'blocks': 8, 'heads': 8},
    {'name': 'patch8_embed256_layers4_heads4', 'patch': 8, 'embed': 256, 'blocks': 4, 'heads': 4},
    {'name': 'patch8_embed256_layers4_heads8', 'patch': 8, 'embed': 256, 'blocks': 4, 'heads': 8},
    {'name': 'patch8_embed256_layers8_heads4', 'patch': 8, 'embed': 256, 'blocks': 8, 'heads': 4},
    {'name': 'patch8_embed256_layers8_heads8', 'patch': 8, 'embed': 256, 'blocks': 8, 'heads': 8},
    {'name': 'patch8_embed512_layers4_heads4', 'patch': 8, 'embed': 512, 'blocks': 4, 'heads': 4},
    {'name': 'patch8_embed512_layers4_heads8', 'patch': 8, 'embed': 512, 'blocks': 4, 'heads': 8},
    {'name': 'patch8_embed512_layers8_heads4', 'patch': 8, 'embed': 512, 'blocks': 8, 'heads': 4},
    {'name': 'patch8_embed512_layers8_heads8', 'patch': 8, 'embed': 512, 'blocks': 8, 'heads': 8}
]

# Class Definitions

In [3]:
class patchEmbedding(nn.Module):
    def __init__(self, patch_size, embed_dim, num_patches, rgb_channels=3):
        super().__init__()
        
        self.num_patches = num_patches
        self.proj = nn.Conv2d(rgb_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2)
        x = x.transpose(1, 2)

        return x

In [4]:
class transformerEncoder(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1):
        super().__init__()
        
        self.layer_norm1 = nn.LayerNorm(embed_dim)
        self.attention = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.layer_norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        x_norm = self.layer_norm1(x)
        output_att, _ = self.attention(x_norm, x_norm, x_norm)
        
        x = x + output_att
        x_norm = self.layer_norm2(x)

        output_mlp = self.mlp(x_norm)
        x = x + output_mlp

        return x

In [5]:
class visionTransformer(nn.Module):
    def __init__(self, image_size, patch_size, num_classes, embed_dim, num_heads, num_layers, mlp_dim, dropout=0.1):
        super().__init__()

        self.num_patches = (image_size // patch_size) ** 2
        self.patch_embed = patchEmbedding(patch_size=patch_size, embed_dim=embed_dim, num_patches=self.num_patches)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        self.dropout = nn.Dropout(dropout)

        self.transformer = nn.ModuleList([transformerEncoder(embed_dim=embed_dim, num_heads=num_heads, mlp_dim=mlp_dim, dropout=dropout) for _ in range(num_layers)])
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls_token = self.cls_token.expand(B, -1, -1)
        
        x = torch.cat((cls_token, x), dim=1)
        x = x + self.pos_embed
        x = self.dropout(x)

        for transformer in self.transformer:
            x = transformer(x)

        x = self.layer_norm(x)
        cls_token_final = x[:, 0]
        x = self.head(cls_token_final)

        return x

## Helper Functions

In [6]:
def build_resnet18(num_classes, pretrained=False):
    weights = torchvision.models.ResNet18_Weights.DEFAULT if pretrained else None
    model = torchvision.models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    return model

In [1]:
def print_results(results):
    print("\n")
    print('=' * 70)
    print(f'Printing Different Model Configuration Results')
    print('=' * 70)
    for cfg in configs:
        print("\n")
        print(f'Patch Size: {cfg['patch']}    |    Embedding Size: {cfg['embed']}    |    Transformer Blocks: {cfg['blocks']}    |    Attention Heads: {cfg['heads']}')
        print('-' * 80)
        
        r = next((res for res in results if res['name'] == cfg['name']), None)
        if r is None:
            continue

        for metric_name, key, fmt in [
        ("Test Accuracy (%)",      "test_acc",  ".4f"),
        ("No. of Parameters",   "num_params",   ".4f"),
        ("FLOPs",            "flops",           ".4f"),
        ("Train Time (s)",  "avg_epoch_time",   ".3f"),
        ]:

            value = r.get(key)            

            try:
                print(f'{metric_name}: {value:{fmt}}')
            except (ValueError, TypeError):
                print(f'{metric_name}: {value}')

In [2]:
def print_baseline(results):
    print("\n")
    print('=' * 70)
    print(f'Printing Baseline ResNet-18 Results')
    print('=' * 70)
    
    for metric_name, key, fmt in [
    ("Test Accuracy (%)",      "test_acc",  ".4f"),
    ("No. of Parameters",   "num_params",   ".4f"),
    ("FLOPs",            "flops",           ".4f"),
    ("Train Time (s)",  "avg_epoch_time",   ".3f"),
    ]:

        value = results.get(key)            

        try:
            print(f'{metric_name}: {value:{fmt}}')
        except (ValueError, TypeError):
            print(f'{metric_name}: {value}')

## Main Function

In [ ]:
def main():

    # Getting cifar100 dataset in DataLoader format
    train_loader, test_loader = load_cifar100(batch_size=batch_size, image_size=image_size)

    # Initializing results list
    results = []

    print(f"Using device: {device}\n")

    print('=' * 60)
    print(f'Starting ResNet-18 Training')
    print('=' * 60)

    resnet_model = build_resnet18(num_classes=num_classes, pretrained=True).to(device)
    resnet_metrics = train_model(model=resnet_model, train_loader=train_loader, test_loader=test_loader, lr=learning_rate, epochs=epochs)
    resnet_metrics['flops'] = get_flops(model=resnet_model, input_size=input_size)
    resnet_metrics['name'] = 'ResNet-18'
    
    for cfg in configs:
        patch = cfg['patch']
        embed = cfg['embed']
        block = cfg['blocks']
        head = cfg['heads']
        mlp = 4 * embed

        print("\n")
        print('=' * 100)
        print(f'Patch Size: {patch}    |    Embedding Size: {embed}    |    Transformer Blocks: {block}    |    Attention Heads: {head}')
        print('=' * 100)

        # Instantiating model variable
        model = visionTransformer(
            image_size=image_size,
            patch_size=patch,
            num_classes=num_classes,
            embed_dim=embed,
            num_heads=head,
            num_layers=block,
            mlp_dim=mlp
        ).to(device)

        metrics = train_model(model=model, train_loader=train_loader, test_loader=test_loader, lr=learning_rate, epochs=epochs)
        metrics['flops'] = get_flops(model=model, input_size=input_size)
        metrics['name'] = cfg['name']

        results.append(metrics)

    # Prinitng Baseline Results
    print_baseline(resnet_metrics)

    # Printing model results    
    print_results(results)

if __name__ == "__main__":
    main()

Starting ResNet-18 Training
Epoch: 1    |    Loss: 2.9213    |    Time: 23.333 sec
Epoch: 2    |    Loss: 2.1463    |    Time: 22.900 sec
Epoch: 3    |    Loss: 1.7501    |    Time: 21.873 sec
Epoch: 4    |    Loss: 1.4008    |    Time: 22.692 sec
Epoch: 5    |    Loss: 1.0653    |    Time: 22.672 sec
Epoch: 6    |    Loss: 0.7390    |    Time: 23.009 sec
Epoch: 7    |    Loss: 0.4739    |    Time: 22.857 sec
Epoch: 8    |    Loss: 0.2639    |    Time: 22.945 sec
Epoch: 9    |    Loss: 0.1415    |    Time: 22.616 sec
Epoch: 10    |    Loss: 0.0857    |    Time: 23.138 sec


Patch Size: 4    |    Embedding Size: 256    |    Transformer Blocks: 4    |    Attention Heads: 4
Epoch: 1    |    Loss: 3.8920    |    Time: 24.457 sec
Epoch: 2    |    Loss: 3.4101    |    Time: 22.263 sec
Epoch: 3    |    Loss: 3.1523    |    Time: 22.128 sec
Epoch: 4    |    Loss: 2.9478    |    Time: 22.003 sec
Epoch: 5    |    Loss: 2.7558    |    Time: 24.208 sec
Epoch: 6    |    Loss: 2.5720    |    Time: 2